<08-5_data_renewal.ipynb>

제미나이 의존도: 30-40%

조기 종료와 가중치 동결 및 해제를 해보고 싶었지만 조기 종료는 에폭이 30~40만큼 엄청 많을 때 하는 거라고 했다.
나는 에폭이 많아봐야 5까지라서 해보지 못했다.
하지만 가중치 동결은 에폭 수와 관련이 없어서 한번 해봤는데, 데이터가 충분히 있고(16000장), 데이터 불균형도 없는 상태라서
오히려 가중치 동결하기 전 모델보다 정확도가 떨어지는 모습을 보았다.
그리고 전에 99.94%가 나온 적이 있는데, 이때 코드의 실수가 조금 있었다.
lr scheduler를.. step()하는 코드는 학습 배치 코드 for문 바깥에 있어야 하는데 나는 for문 안에 넣어버렸다.
불행 중 다행(?)인 건 T_max의 값이 50이라서 우연히 가중치는 최저점을 향해 가고 있었기 때문에 Loss가 낮았던 것이다.

지금 코드에서는 T_max도 값을 EPOCHS으로 바꾸고 scheduler.step()도 for문 바깥에 두었더니 결과는 대단했다.
5번의 폴드 중에서 정확도 100%가 두 번이나 나왔다. (T_max 값 잘 설정하자.. scheduler.step() 잘 끼워넣자..)

다음 챕터는.. 예전에도 한번 해본 적이 있는 WeightedRandomSampler이다.
데이터가 불균형(정상 데이터 많음, 불량 데이터 적음)일 때 쓰는 건데, 
정상 데이터는 적게 뽑고 불량 데이터는 많이 뽑음으로써 과적합을 막는것이다.
렛츠기릿.

In [1]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold

fruits = ["apple", "banana", "orange"]
classes = []
data = []

N_SPLITS = 5
BATCH_SIZE = 64

for fruit in fruits:
    path = f"images/kaggle_data/dataset/{fruit}"
    categories = [f"fresh_{fruit}", f"rotten_{fruit}"]
    classes.extend(categories)

    for category in categories:
        folder_path = os.path.join(path, category)
        for image in os.listdir(folder_path):
            if image.endswith(("jpg", "jpeg", "png")):
                data.append({
                    "image_path": os.path.join(folder_path, image), 
                    "label_name": category
                })

df = pd.DataFrame(data)
# print(len(df))
label_map = {name: i for i, name in enumerate(classes)}
df['label'] = df['label_name'].map(label_map)

train_val_df, test_df = train_test_split(
    df, stratify=df['label'], test_size=0.1, random_state=42, shuffle=True
)

train_val_df = train_val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

skf = StratifiedKFold(n_splits=N_SPLITS, random_state=42, shuffle=True)

train_val_df['kfold'] = -1
for fold_idx, (train_idx, val_idx) in enumerate(skf.split(train_val_df, train_val_df['label'])):
    train_val_df.loc[val_idx, 'kfold'] = fold_idx

In [8]:
from torch.utils.data import Dataset, DataLoader
import cv2

class FruitsDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = row['image_path']
        image = cv2.imread(image)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if self.transform:
            aug = self.transform(image=image)
            image = aug['image']

        label = int(row['label'])
        return image, torch.tensor(label, dtype=torch.long)

In [4]:
import albumentations as A
from albumentations.pytorch import ToTensorV2

train_transform = A.Compose([
    A.Resize(128, 128), 
    A.HorizontalFlip(p=0.5), 
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)), 
    ToTensorV2()
])

val_test_transform = A.Compose([
    A.Resize(128, 128), 
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)), 
    ToTensorV2()
])

In [9]:
import torch

device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')

test_dataset = FruitsDataset(df=test_df, transform=val_test_transform)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [19]:
import torchvision.models as models
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
EPOCHS = 3

for fold in range(N_SPLITS):
    print(f"\n===FOLD {fold} START===")
    train_df = train_val_df[train_val_df['kfold'] != fold].reset_index(drop=True)
    val_df = train_val_df[train_val_df['kfold'] == fold].reset_index(drop=True)

    train_dataset = FruitsDataset(df=train_df, transform=train_transform)
    val_dataset = FruitsDataset(df=val_df, transform=val_test_transform)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

    # 모든 가중치 동결
    # for param in model.parameters():
    #     param.requires_grad = False
    
    model.fc = nn.Linear(model.fc.in_features, len(classes))
    model = model.to(device)
    
    criterion = nn.CrossEntropyLoss()

    # 학습 가능한 파라미터만 필터링해서 optimizer에 전달
    # optimizer = optim.Adam([p for p in model.parameters() if p.requires_grad], lr=0.001)
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

    best_val_loss = float('inf')
    for epoch in range(EPOCHS):
        # === Train Start ===
        model.train()
        train_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * images.size(0)

        # scheduler.step()은 배치 돌리는 for문 밖에 나와야 한다.
        scheduler.step()
        current_lr = scheduler.get_last_lr()[0]
        print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss/len(train_dataset):.4f} | lr = {current_lr:.6f}")
        # === Train End ===
        
        # === Val Start ===
        model.eval()
        val_loss = 0.0
        val_count = 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * images.size(0)
                preds = outputs.argmax(1)
                val_count += (preds == labels).sum().item()

        val_loss /= len(val_dataset)
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), f"best_model__{fold}.pth")

        print(f"Val Loss: {val_loss:.4f} | Acc: {val_count/len(val_dataset) * 100:.2f}%")
        # === Val End ===
                
    # === Test Start ===
    best_model = models.resnet18(weights=None)
    best_model.fc = nn.Linear(best_model.fc.in_features, len(classes))
    best_model.load_state_dict(torch.load(f"best_model__{fold}.pth"))
    best_model.to(device)
    best_model.eval()

    test_count = 0.0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = best_model(images)
            loss = criterion(outputs, labels)

            preds = outputs.argmax(1)
            test_count += (preds == labels).sum().item()

    print(f"Test Acc: {test_count/len(test_dataset) * 100:.2f}%")
    # === Test End ===


===FOLD 0 START===
Epoch 1/3 | Train Loss: 0.1588 | lr = 0.000750
Val Loss: 0.1416 | Acc: 94.82%
Epoch 2/3 | Train Loss: 0.0500 | lr = 0.000251
Val Loss: 0.0225 | Acc: 99.31%
Epoch 3/3 | Train Loss: 0.0117 | lr = 0.000001
Val Loss: 0.0017 | Acc: 100.00%
Test Acc: 99.94%

===FOLD 1 START===
Epoch 1/3 | Train Loss: 0.1682 | lr = 0.000750
Val Loss: 0.3271 | Acc: 89.41%
Epoch 2/3 | Train Loss: 0.0608 | lr = 0.000251
Val Loss: 0.0427 | Acc: 98.58%
Epoch 3/3 | Train Loss: 0.0078 | lr = 0.000001
Val Loss: 0.0023 | Acc: 99.93%
Test Acc: 100.00%

===FOLD 2 START===
Epoch 1/3 | Train Loss: 0.1739 | lr = 0.000750
Val Loss: 0.0637 | Acc: 97.82%
Epoch 2/3 | Train Loss: 0.0560 | lr = 0.000251
Val Loss: 0.0728 | Acc: 97.16%
Epoch 3/3 | Train Loss: 0.0152 | lr = 0.000001
Val Loss: 0.0052 | Acc: 99.87%
Test Acc: 99.76%

===FOLD 3 START===
Epoch 1/3 | Train Loss: 0.1674 | lr = 0.000750
Val Loss: 0.2813 | Acc: 91.58%
Epoch 2/3 | Train Loss: 0.0724 | lr = 0.000251
Val Loss: 0.0324 | Acc: 99.11%
Epoch 3/3